## 리스트(List)와 셋(Set): 탐색 속도의 차이

데이터 전처리 과정에서 가장 빈번하게 하는 작업은 **"이 데이터가 저기에 포함되어 있는가?"** 를 확인하는 것이다.

예를 들어, 100만 개의 불량품 ID 리스트(`bad_ids`)가 있고, 현재 검사 중인 제품 ID가 불량인지 확인한다고 가정해 보자.

In [1]:
import time

# 1억개의 데이터가 있는 bad_ids_list 생성: 약12초 소요
bad_ids_list = [f"id_{i}" for i in range(100_000_000)]
current_id = "id_99999999"  # 검색할 ID

In [2]:
# [Bad] 리스트를 사용한 검색
# 리스트는 처음부터 끝까지 하나씩 뒤져야 한다. (O(n))
# 소요시간 측정
start_time = time.time()

if current_id in bad_ids_list:
    print("불량입니다.")

end_time = time.time()
print("소요시간:", end_time - start_time)

불량입니다.
소요시간: 1.5145800113677979


리스트에서 특정 값을 찾는 연산(`in`)은 최악의 경우 리스트 전체를 훑어야 한다. 데이터가 늘어날수록 속도는 정직하게 느려진다. 이를 시간 복잡도로 **$O(n)$** 이라고 한다.

반면, **셋(Set)** 은 순서가 없는 대신 해시(Hash) 알고리즘을 사용한다. 데이터가 100만 개든 1,000만 개든, 찾는 값이 어디 있는지 즉시 알아낸다.

In [3]:
# 리스트를 셋으로 변환 : 변환시간은 오래 걸린다. 
bad_ids_set = set(bad_ids_list)  # 리스트를 셋으로 변환

In [4]:
# [Good] 셋을 사용한 검색
# 셋은 데이터 양과 상관없이 거의 즉시 찾는다. (O(1))
start_time = time.time()
if current_id in bad_ids_set:
    print("불량입니다.")

end_time = time.time()
print("소요시간:", end_time - start_time)

불량입니다.
소요시간: 0.0008668899536132812


**결론:** 순서가 중요하지 않고, **중복을 제거**하거나 **포함 여부(Membership Test)**를 여러 번 검사해야 한다면 무조건 `Set`을 사용한다.

## 튜플(Tuple): 불변의 약속

`List`와 `Tuple`은 비슷해 보이지만 결정적인 차이가 있다. 리스트는 내용을 바꿀 수 있고(Mutable), 튜플은 바꿀 수 없다(Immutable).

연구 데이터, 특히 설정값(Config)이나 머신러닝 모델의 하이퍼파라미터처럼 **절대 변하면 안 되는 값**들은 리스트가 아닌 튜플에 담아야 한다.

1.  **안전성:** 실수로 데이터를 수정하는 것을 코드로 원천 봉쇄한다.
2.  **성능:** 튜플은 리스트보다 메모리를 적게 차지하고, 생성 속도가 미세하게 더 빠르다.

In [ ]:
# [Bad] 좌표나 설정값을 리스트로 관리
image_size = [1920, 1080] 

print(image_size) # [1920, 1080]

[1920, 1080]


In [7]:
# [Good] 변하지 않는 값은 튜플로 관리
image_size = (1920, 1080)
print(image_size) # (1920, 1080)

(1920, 1080)


## 딕셔너리와 Counter: 개수 세기의 달인

텍스트 마이닝이나 범주형 데이터를 다룰 때, 각 항목이 몇 번 등장했는지 세는(Counting) 작업은 필수적이다. 초심자들은 보통 딕셔너리와 for 문을 조합해서 짠다.

In [8]:
# [Normal] 일반적인 딕셔너리 사용
words = ['apple', 'banana', 'apple', 'cherry', 'banana', 'banana']
counts = {}

for word in words:
    if word in counts:
        counts[word] += 1
    else:
        counts[word] = 1

print(counts) # {'apple': 2, 'banana': 3, 'cherry': 1}

{'apple': 2, 'banana': 3, 'cherry': 1}


하지만 파이썬의 `collections` 모듈에 있는 `Counter`를 사용하면 이 모든 과정을 한 줄로 줄일 수 있다. 이것이 진정한 파이썬다운 코드다.

In [9]:
# [Pythonic] Counter 사용
from collections import Counter

words = ['apple', 'banana', 'apple', 'cherry', 'banana', 'banana']
counts = Counter(words)

print(counts)
# 출력: Counter({'banana': 3, 'apple': 2, 'cherry': 1})

# 가장 많이 등장한 2개 추출
print(counts.most_common(2))

Counter({'banana': 3, 'apple': 2, 'cherry': 1})
[('banana', 3), ('apple', 2)]


## TIP : 시간 복잡도에 대한 감각

현업에서 주니어 데이터 사이언티스트들이 가장 많이 저지르는 실수가 바로 **'이중 for 문(Double Loop)'**입니다.

```python
# A리스트의 ID가 B리스트에 있는지 확인하려고 이중 반복문을 돌립니다.
# A가 10만 개, B가 10만 개라면 연산 횟수는 100억 번(100,000 * 100,000)이 됩니다.
for a in list_A:
    for b in list_B:
        if a == b:
            # ...
```

이런 코드는 슈퍼컴퓨터를 가져와도 몇 시간이 걸립니다.
만약 `list_B`를 `set(list_B)`로 바꾸기만 해도, 이 로직은 100억 번 연산에서 10만 번 연산으로 줄어듭니다. 시간으로 치면 **3시간 걸릴 작업이 0.1초 만에 끝나는 마법** 입니다.

자료구조를 선택할 때는 항상 **"데이터가 커졌을 때 이 코드가 얼마나 느려질까?"** 를 고민하는 습관을 들여야 합니다. 그것이 엔지니어링 감각입니다.

In [10]:
import random
import time
from typing import Iterable, Sequence, Tuple


def _generate_lists(size: int, seed: int = 42) -> Tuple[list[int], list[int]]:
    rng = random.Random(seed)
    list_a = [rng.randrange(size * 2) for _ in range(size)]
    list_b = [rng.randrange(size * 2) for _ in range(size)]
    return list_a, list_b


def _count_matches_nested(a_list: Sequence[int], b_iter: Iterable[int]) -> int:
    count = 0
    for a in a_list:
        for b in b_iter:
            if a == b:
                count += 1
    return count


def _count_matches_membership(a_list: Sequence[int], b_set: set[int]) -> int:
    count = 0
    for a in a_list:
        if a in b_set:
            count += 1
    return count


def measure_nested_list(list_a: Sequence[int], list_b: Sequence[int]) -> float:
    start = time.perf_counter()
    _count_matches_nested(list_a, list_b)
    return time.perf_counter() - start


def measure_set_membership(list_a: Sequence[int], list_b: Sequence[int]) -> float:
    set_b = set(list_b)
    start = time.perf_counter()
    _count_matches_membership(list_a, set_b)
    return time.perf_counter() - start


def main(size: int = 20000) -> None:
    list_a, list_b = _generate_lists(size)

    time_list = measure_nested_list(list_a, list_b)
    time_set = measure_set_membership(list_a, list_b)

    print(f"size={size}")
    print(f"nested list: {time_list:.6f}s")
    print(f"set membership: {time_set:.6f}s")

In [11]:
main()

size=20000
nested list: 5.953548s
set membership: 0.000766s
